In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🚛 Scania APS Failure Prediction\n",
    "## 🔧 Feature Engineering & Preprocessing Pipeline\n",
    "\n",
    "**Author**: Data Scientist  \n",
    "**Date**: 2026  \n",
    "**Purpose**: Transform raw sensor data into predictive features optimized for business cost minimization\n",
    "\n",
    "---\n",
    "\n",
    "## 📋 Table of Contents\n",
    "1. [Setup & Data Loading](#setup--data-loading)\n",
    "2. [Missing Value Treatment](#missing-value-treatment)\n",
    "3. [Histogram Feature Engineering](#histogram-feature-engineering)\n",
    "4. [Counter Feature Engineering](#counter-feature-engineering)\n",
    "5. [Indicator Features](#indicator-features)\n",
    "6. [Feature Transformation](#feature-transformation)\n",
    "7. [Feature Selection](#feature-selection)\n",
    "8. [Preprocessing Pipeline](#preprocessing-pipeline)\n",
    "9. [Save Processed Data](#save-processed-data)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer\n",
    "from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif\n",
    "from sklearn.impute import SimpleImputer, KNNImputer\n",
    "from sklearn.experimental import enable_iterative_imputer\n",
    "from sklearn.impute import IterativeImputer\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.compose import ColumnTransformer\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "\n",
    "# Custom color palette\n",
    "COLORS = {\n",
    "    'primary': '#003366',\n",
    "    'secondary': '#FFB612',\n",
    "    'success': '#28A745',\n",
    "    'danger': '#DC3545'\n",
    "}"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📂 Setup & Data Loading"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load data from Phase 1\n",
    "train_df = pd.read_csv('../data/raw/train.csv')\n",
    "test_df = pd.read_csv('../data/raw/test.csv')\n",
    "\n",
    "# Separate features and target\n",
    "X_train_raw = train_df.drop('class', axis=1)\n",
    "y_train = train_df['class'].map({'neg': 0, 'pos': 1})  # Encode target: 0=neg, 1=pos\n",
    "\n",
    "X_test_raw = test_df.drop('class', axis=1) if 'class' in test_df.columns else test_df\n",
    "y_test = test_df['class'].map({'neg': 0, 'pos': 1}) if 'class' in test_df.columns else None\n",
    "\n",
    "print(\"📊 Data loaded successfully!\")\n",
    "print(f\"Training features: {X_train_raw.shape}\")\n",
    "print(f\"Training target: {y_train.shape}\")\n",
    "print(f\"Test features: {X_test_raw.shape}\")\n",
    "if y_test is not None:\n",
    "    print(f\"Test target: {y_test.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🔍 Missing Value Treatment\n",
    "\n",
    "### Strategy Based on EDA Findings:\n",
    "1. **High missing (>80%)**: Drop or create binary indicator\n",
    "2. **Medium missing (20-80%)**: Advanced imputation (MICE)\n",
    "3. **Low missing (<20%)**: Simple imputation with median"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def analyze_missing(df, threshold=0.8):\n",
    "    \"\"\"Analyze missing values and categorize features\"\"\"\n",
    "    missing_pct = df.isna().sum() / len(df) * 100\n",
    "    \n",
    "    high_missing = missing_pct[missing_pct >= threshold].index.tolist()\n",
    "    medium_missing = missing_pct[(missing_pct >= 20) & (missing_pct < threshold)].index.tolist()\n",
    "    low_missing = missing_pct[(missing_pct > 0) & (missing_pct < 20)].index.tolist()\n",
    "    no_missing = missing_pct[missing_pct == 0].index.tolist()\n",
    "    \n",
    "    return {\n",
    "        'high': high_missing,\n",
    "        'medium': medium_missing,\n",
    "        'low': low_missing,\n",
    "        'none': no_missing\n",
    "    }\n",
    "\n",
    "# Analyze missing patterns\n",
    "missing_categories = analyze_missing(X_train_raw)\n",
    "\n",
    "print(\"📊 Missing Value Categories:\")\n",
    "print(f\"  High missing (>80%): {len(missing_categories['high'])} features\")\n",
    "print(f\"  Medium missing (20-80%): {len(missing_categories['medium'])} features\")\n",
    "print(f\"  Low missing (<20%): {len(missing_categories['low'])} features\")\n",
    "print(f\"  No missing: {len(missing_categories['none'])} features\")\n",
    "\n",
    "if missing_categories['high']:\n",
    "    print(f\"\\n⚠️ High missing features: {missing_categories['high'][:10]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class MissingValueHandler:\n",
    "    \"\"\"Advanced missing value handler with multiple strategies\"\"\"\n",
    "    \n",
    "    def __init__(self, high_threshold=0.8):\n",
    "        self.high_threshold = high_threshold\n",
    "        self.feature_groups = {}\n",
    "        self.imputers = {}\n",
    "        self.binary_flags = {}\n",
    "    \n",
    "    def fit(self, X):\n",
    "        \"\"\"Fit missing value handler to data\"\"\"\n",
    "        missing_pct = X.isna().sum() / len(X) * 100\n",
    "        \n",
    "        # Categorize features\n",
    "        self.feature_groups['high'] = missing_pct[missing_pct >= self.high_threshold].index.tolist()\n",
    "        self.feature_groups['medium'] = missing_pct[(missing_pct >= 20) & (missing_pct < self.high_threshold)].index.tolist()\n",
    "        self.feature_groups['low'] = missing_pct[(missing_pct > 0) & (missing_pct < 20)].index.tolist()\n",
    "        self.feature_groups['none'] = missing_pct[missing_pct == 0].index.tolist()\n",
    "        \n",
    "        # Create imputers\n",
    "        if self.feature_groups['medium']:\n",
    "            self.imputers['medium'] = IterativeImputer(\n",
    "                estimator=None,\n",
    "                max_iter=10,\n",
    "                random_state=42,\n",
    "                n_nearest_features=5\n",
    "            )\n",
    "            self.imputers['medium'].fit(X[self.feature_groups['medium']])\n",
    "        \n",
    "        if self.feature_groups['low']:\n",
    "            self.imputers['low'] = SimpleImputer(strategy='median')\n",
    "            self.imputers['low'].fit(X[self.feature_groups['low']])\n",
    "        \n",
    "        return self\n",
    "    \n",
    "    def transform(self, X):\n",
    "        \"\"\"Transform data with missing value handling\"\"\"\n",
    "        X_transformed = X.copy()\n",
    "        \n",
    "        # 1. Handle high missing: Create binary flags\n",
    "        for feature in self.feature_groups['high']:\n",
    "            X_transformed[f'{feature}_missing_flag'] = X[feature].isna().astype(int)\n",
    "        \n",
    "        # 2. Handle medium missing: MICE imputation\n",
    "        if self.feature_groups['medium']:\n",
    "            X_transformed[self.feature_groups['medium']] = self.imputers['medium'].transform(\n",
    "                X_transformed[self.feature_groups['medium']]\n",
    "            )\n",
    "        \n",
    "        # 3. Handle low missing: Median imputation\n",
    "        if self.feature_groups['low']:\n",
    "            X_transformed[self.feature_groups['low']] = self.imputers['low'].transform(\n",
    "                X_transformed[self.feature_groups['low']]\n",
    "            )\n",
    "        \n",
    "        return X_transformed\n",
    "    \n",
    "    def fit_transform(self, X):\n",
    "        \"\"\"Fit and transform in one step\"\"\"\n",
    "        return self.fit(X).transform(X)\n",
    "\n",
    "# Apply missing value handler\n",
    "missing_handler = MissingValueHandler(high_threshold=0.8)\n",
    "X_train_imputed = missing_handler.fit_transform(X_train_raw)\n",
    "X_test_imputed = missing_handler.transform(X_test_raw)\n",
    "\n",
    "print(f\"✅ Missing value handling complete!\")\n",
    "print(f\"Training shape: {X_train_imputed.shape}\")\n",
    "print(f\"Test shape: {X_test_imputed.shape}\")\n",
    "print(f\"Added {X_train_imputed.shape[1] - X_train_raw.shape[1]} missing flag features\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Missing Value Strategy\n",
    "\n",
    "**Rationale**:\n",
    "1. **Missing flags**: Missingness often contains signal - sensors that fail may indicate other issues\n",
    "2. **MICE imputation**: Uses multiple features to predict missing values, preserving relationships\n",
    "3. **Median imputation**: Robust to outliers and preserves central tendency\n",
    "\n",
    "**Business Impact**:\n",
    "- Proper handling prevents information loss\n",
    "- Missing flags may capture early warning signs of APS issues"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Histogram Feature Engineering\n",
    "\n",
    "Histogram features (ay_000-ay_009) represent distributions. We'll extract statistical features."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class HistogramFeatureEngineer:\n",
    "    \"\"\"Extract statistical features from histogram data\"\"\"\n",
    "    \n",
    "    def __init__(self, prefix='ay_'):\n",
    "        self.prefix = prefix\n",
    "        self.hist_features = []\n",
    "    \n",
    "    def fit(self, X):\n",
    "        \"\"\"Identify histogram features\"\"\"\n",
    "        self.hist_features = [col for col in X.columns if col.startswith(self.prefix)]\n",
    "        return self\n",
    "    \n",
    "    def transform(self, X):\n",
    "        \"\"\"Extract statistical features from histograms\"\"\"\n",
    "        X_transformed = X.copy()\n",
    "        \n",
    "        if not self.hist_features:\n",
    "            return X_transformed\n",
    "        \n",
    "        # Convert hist features to numpy array for efficient computation\n",
    "        hist_data = X[self.hist_features].values\n",
    "        \n",
    "        # Create statistical features\n",
    "        X_transformed[f'{self.prefix}mean'] = np.nanmean(hist_data, axis=1)\n",
    "        X_transformed[f'{self.prefix}variance'] = np.nanvar(hist_data, axis=1)\n",
    "        X_transformed[f'{self.prefix}std'] = np.nanstd(hist_data, axis=1)\n",
    "        X_transformed[f'{self.prefix}skew'] = pd.DataFrame(hist_data).skew(axis=1)\n",
    "        X_transformed[f'{self.prefix}kurtosis'] = pd.DataFrame(hist_data).kurtosis(axis=1)\n",
    "        \n",
    "        # Find peak location (mode)\n",
    "        peak_indices = np.nanargmax(hist_data, axis=1)\n",
    "        X_transformed[f'{self.prefix}peak_index'] = peak_indices\n",
    "        \n",
    "        # Calculate concentration (entropy-based)\n",
    "        hist_probs = hist_data / np.nansum(hist_data, axis=1, keepdims=True)\n",
    "        entropy = -np.nansum(hist_probs * np.log(hist_probs + 1e-10), axis=1)\n",
    "        X_transformed[f'{self.prefix}entropy'] = entropy\n",
    "        \n",
    "        # Calculate spread (interquartile range of distribution)\n",
    "        hist_quantiles = np.percentile(hist_data, [25, 75], axis=1)\n",
    "        X_transformed[f'{self.prefix}iqr'] = hist_quantiles[1] - hist_quantiles[0]\n",
    "        \n",
    "        return X_transformed\n",
    "    \n",
    "    def fit_transform(self, X):\n",
    "        \"\"\"Fit and transform in one step\"\"\"\n",
    "        return self.fit(X).transform(X)\n",
    "\n",
    "# Apply histogram feature engineering\n",
    "hist_engineer = HistogramFeatureEngineer(prefix='ay_')\n",
    "X_train_hist = hist_engineer.fit_transform(X_train_imputed)\n",
    "X_test_hist = hist_engineer.transform(X_test_imputed)\n",
    "\n",
    "print(f\"✅ Histogram features engineered!\")\n",
    "print(f\"Added {X_train_hist.shape[1] - X_train_imputed.shape[1]} histogram-derived features\")\n",
    "print(f\"New shape: {X_train_hist.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Histogram Features\n",
    "\n",
    "**Engineered Features**:\n",
    "- **Mean**: Average pressure/signal level\n",
    "- **Variance/Std**: Stability of pressure\n",
    "- **Skewness**: Asymmetry in distribution\n",
    "- **Peak Index**: Most common pressure level\n",
    "- **Entropy**: Complexity of pressure pattern\n",
    "- **IQR**: Spread of pressure readings\n",
    "\n",
    "**Business Value**:\n",
    "- APS failures often show pressure instability or loss\n",
    "- These features capture subtle changes in pressure distribution\n",
    "- Early warning indicators for maintenance"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📈 Counter Feature Engineering\n",
    "\n",
    "Counter features (ag_000-ag_009, az_000-az_009, etc.) represent counts of events. We'll create rate-of-change features."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class CounterFeatureEngineer:\n",
    "    \"\"\"Engineer features from counter data\"\"\"\n",
    "    \n",
    "    def __init__(self, counter_groups=['ag_', 'az_', 'ba_', 'cs_', 'ee_']):\n",
    "        self.counter_groups = counter_groups\n",
    "        self.groups = {}\n",
    "    \n",
    "    def fit(self, X):\n",
    "        \"\"\"Identify counter feature groups\"\"\"\n",
    "        for prefix in self.counter_groups:\n",
    "            self.groups[prefix] = [col for col in X.columns if col.startswith(prefix)]\n",
    "        return self\n",
    "    \n",
    "    def transform(self, X):\n",
    "        \"\"\"Create rate-of-change and cumulative features\"\"\"\n",
    "        X_transformed = X.copy()\n",
    "        \n",
    "        for prefix, features in self.groups.items():\n",
    "            if len(features) > 1:\n",
    "                # Create cumulative sum\n",
    "                X_transformed[f'{prefix}cumsum'] = X[features].sum(axis=1)\n",
    "                \n",
    "                # Create rate of change (difference between consecutive)\n",
    "                for i in range(len(features) - 1):\n",
    "                    diff = X[features[i+1]] - X[features[i]]\n",
    "                    X_transformed[f'{prefix}diff_{i}'] = diff.clip(lower=0)  # Counters should only increase\n",
    "                \n",
    "                # Create max/min ratios\n",
    "                X_transformed[f'{prefix}max_min_ratio'] = X[features].max(axis=1) / (X[features].min(axis=1) + 1e-10)\n",
    "        \n",
    "        return X_transformed\n",
    "    \n",
    "    def fit_transform(self, X):\n",
    "        \"\"\"Fit and transform in one step\"\"\"\n",
    "        return self.fit(X).transform(X)\n",
    "\n",
    "# Apply counter feature engineering\n",
    "counter_engineer = CounterFeatureEngineer()\n",
    "X_train_counter = counter_engineer.fit_transform(X_train_hist)\n",
    "X_test_counter = counter_engineer.transform(X_test_hist)\n",
    "\n",
    "print(f\"✅ Counter features engineered!\")\n",
    "print(f\"Added {X_train_counter.shape[1] - X_train_hist.shape[1]} counter-derived features\")\n",
    "print(f\"New shape: {X_train_counter.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Counter Features\n",
    "\n",
    "**Engineered Features**:\n",
    "- **Cumulative Sum**: Total events across sensors\n",
    "- **Rate of Change**: How quickly events accumulate\n",
    "- **Max/Min Ratio**: Extremes in event patterns\n",
    "\n",
    "**Business Value**:\n",
    "- Rapidly increasing counters may indicate component wear\n",
    "- Abrupt changes in rates suggest system stress\n",
    "- Cumulative totals help track component lifecycle"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🎯 Feature Transformation\n",
    "\n",
    "Apply transformations to handle skewness and scaling."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class FeatureTransformer:\n",
    "    \"\"\"Apply transformations to features\"\"\"\n",
    "    \n",
    "    def __init__(self):\n",
    "        self.scaler = RobustScaler()  # Robust to outliers\n",
    "        self.transformer = None\n",
    "        self.features_to_transform = []\n",
    "    \n",
    "    def fit(self, X):\n",
    "        \"\"\"Fit transformers to data\"\"\"\n",
    "        # Identify skewed features (skewness > 1 or < -1)\n",
    "        skewness = X.skew()\n",
    "        self.features_to_transform = skewness[(skewness > 1) | (skewness < -1)].index.tolist()\n",
    "        \n",
    "        # Apply PowerTransformer to skewed features\n",
    "        if self.features_to_transform:\n",
    "            self.transformer = PowerTransformer(method='yeo-johnson')\n",
    "            self.transformer.fit(X[self.features_to_transform])\n",
    "        \n",
    "        # Fit scaler\n",
    "        self.scaler.fit(X)\n",
    "        \n",
    "        return self\n",
    "    \n",
    "    def transform(self, X):\n",
    "        \"\"\"Transform data\"\"\"\n",
    "        X_transformed = X.copy()\n",
    "        \n",
    "        # Transform skewed features\n",
    "        if self.transformer and self.features_to_transform:\n",
    "            X_transformed[self.features_to_transform] = self.transformer.transform(\n",
    "                X_transformed[self.features_to_transform]\n",
    "            )\n",
    "        \n",
    "        # Scale all features\n",
    "        X_transformed = pd.DataFrame(\n",
    "            self.scaler.transform(X_transformed),\n",
    "            columns=X.columns,\n",
    "            index=X.index\n",
    "        )\n",
    "        \n",
    "        return X_transformed\n",
    "    \n",
    "    def fit_transform(self, X):\n",
    "        \"\"\"Fit and transform in one step\"\"\"\n",
    "        return self.fit(X).transform(X)\n",
    "\n",
    "# Apply feature transformation\n",
    "transformer = FeatureTransformer()\n",
    "X_train_transformed = transformer.fit_transform(X_train_counter)\n",
    "X_test_transformed = transformer.transform(X_test_counter)\n",
    "\n",
    "print(f\"✅ Feature transformation complete!\")\n",
    "print(f\"Transformed {len(transformer.features_to_transform)} skewed features\")\n",
    "print(f\"Final shape: {X_train_transformed.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🎯 Feature Selection\n",
    "\n",
    "Select the most informative features while maintaining computational efficiency."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class FeatureSelector:\n",
    "    \"\"\"Feature selection with multiple strategies\"\"\"\n",
    "    \n",
    "    def __init__(self, variance_threshold=0.01, k_features=50):\n",
    "        self.variance_threshold = variance_threshold\n",
    "        self.k_features = k_features\n",
    "        self.selected_features = []\n",
    "        self.selector = None\n",
    "    \n",
    "    def fit(self, X, y):\n",
    "        \"\"\"Fit feature selector\"\"\"\n",
    "        # 1. Remove low variance features\n",
    "        variance_filter = VarianceThreshold(threshold=self.variance_threshold)\n",
    "        variance_filter.fit(X)\n",
    "        high_variance_features = X.columns[variance_filter.get_support()].tolist()\n",
    "        \n",
    "        # 2. Select top k features using mutual information\n",
    "        self.selector = SelectKBest(mutual_info_classif, k=min(self.k_features, len(high_variance_features)))\n",
    "        self.selector.fit(X[high_variance_features], y)\n",
    "        \n",
    "        # Get selected features\n",
    "        selected_indices = self.selector.get_support(indices=True)\n",
    "        self.selected_features = [high_variance_features[i] for i in selected_indices]\n",
    "        \n",
    "        return self\n",
    "    \n",
    "    def transform(self, X):\n",
    "        \"\"\"Transform data with selected features\"\"\"\n",
    "        return X[self.selected_features]\n",
    "    \n",
    "    def fit_transform(self, X, y):\n",
    "        \"\"\"Fit and transform in one step\"\"\"\n",
    "        return self.fit(X, y).transform(X)\n",
    "\n",
    "# Apply feature selection\n",
    "feature_selector = FeatureSelector(variance_threshold=0.01, k_features=100)\n",
    "X_train_selected = feature_selector.fit_transform(X_train_transformed, y_train)\n",
    "X_test_selected = feature_selector.transform(X_test_transformed)\n",
    "\n",
    "print(f\"✅ Feature selection complete!\")\n",
    "print(f\"Selected {len(feature_selector.selected_features)} features\")\n",
    "print(f\"Final shape: {X_train_selected.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze selected features\n",
    "selected_feature_info = pd.DataFrame({\n",
    "    'Feature': feature_selector.selected_features,\n",
    "    'Score': feature_selector.selector.scores_[feature_selector.selector.get_support()]\n",
    "}).sort_values('Score', ascending=False)\n",
    "\n",
    "print(\"📊 TOP 20 SELECTED FEATURES:\")\n",
    "print(\"=\"*60)\n",
    "for idx, row in selected_feature_info.head(20).iterrows():\n",
    "    print(f\"  {row['Feature']}: {row['Score']:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Feature Selection\n",
    "\n",
    "**Selection Strategy**:\n",
    "1. **Variance Threshold**: Remove features that don't vary enough to be predictive\n",
    "2. **Mutual Information**: Select features with strongest relationship to target\n",
    "3. **Balance**: Keep enough features for signal while reducing noise\n",
    "\n",
    "**Business Value**:\n",
    "- Focus on most impactful sensors/components\n",
    "- Reduce model complexity and inference time\n",
    "- Identify critical monitoring points for maintenance"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🔧 Preprocessing Pipeline\n",
    "\n",
    "Create a complete preprocessing pipeline for production use."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class PreprocessingPipeline:\n",
    "    \"\"\"Complete preprocessing pipeline for production\"\"\"\n",
    "    \n",
    "    def __init__(self):\n",
    "        self.missing_handler = None\n",
    "        self.hist_engineer = None\n",
    "        self.counter_engineer = None\n",
    "        self.transformer = None\n",
    "        self.selector = None\n",
    "    \n",
    "    def fit(self, X, y=None):\n",
    "        \"\"\"Fit all preprocessing components\"\"\"\n",
    "        self.missing_handler = MissingValueHandler(high_threshold=0.8)\n",
    "        self.missing_handler.fit(X)\n",
    "        \n",
    "        X_imputed = self.missing_handler.transform(X)\n",
    "        \n",
    "        self.hist_engineer = HistogramFeatureEngineer(prefix='ay_')\n",
    "        self.hist_engineer.fit(X_imputed)\n",
    "        \n",
    "        X_hist = self.hist_engineer.transform(X_imputed)\n",
    "        \n",
    "        self.counter_engineer = CounterFeatureEngineer()\n",
    "        self.counter_engineer.fit(X_hist)\n",
    "        \n",
    "        X_counter = self.counter_engineer.transform(X_hist)\n",
    "        \n",
    "        self.transformer = FeatureTransformer()\n",
    "        self.transformer.fit(X_counter)\n",
    "        \n",
    "        X_transformed = self.transformer.transform(X_counter)\n",
    "        \n",
    "        self.selector = FeatureSelector(variance_threshold=0.01, k_features=100)\n",
    "        if y is not None:\n",
    "            self.selector.fit(X_transformed, y)\n",
    "        else:\n",
    "            # If no y, use all features\n",
    "            self.selector.selected_features = X_transformed.columns.tolist()\n",
    "        \n",
    "        return self\n",
    "    \n",
    "    def transform(self, X):\n",
    "        \"\"\"Transform data using fitted pipeline\"\"\"\n",
    "        X_imputed = self.missing_handler.transform(X)\n",
    "        X_hist = self.hist_engineer.transform(X_imputed)\n",
    "        X_counter = self.counter_engineer.transform(X_hist)\n",
    "        X_transformed = self.transformer.transform(X_counter)\n",
    "        X_selected = self.selector.transform(X_transformed)\n",
    "        \n",
    "        return X_selected\n",
    "    \n",
    "    def fit_transform(self, X, y=None):\n",
    "        \"\"\"Fit and transform in one step\"\"\"\n",
    "        return self.fit(X, y).transform(X)\n",
    "\n",
    "# Create and apply pipeline\n",
    "preprocessing_pipeline = PreprocessingPipeline()\n",
    "X_train_processed = preprocessing_pipeline.fit_transform(X_train_raw, y_train)\n",
    "X_test_processed = preprocessing_pipeline.transform(X_test_raw)\n",
    "\n",
    "print(f\"✅ Preprocessing pipeline complete!\")\n",
    "print(f\"Training processed shape: {X_train_processed.shape}\")\n",
    "print(f\"Test processed shape: {X_test_processed.shape}\")\n",
    "print(f\"Final feature count: {X_train_processed.shape[1]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 💾 Save Processed Data\n",
    "\n",
    "Save all processed data for Phase 3 (Model Development)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save processed data\n",
    "import joblib\n",
    "\n",
    "# Create processed directory if it doesn't exist\n",
    "import os\n",
    "os.makedirs('../data/processed', exist_ok=True)\n",
    "\n",
    "# Save processed features\n",
    "joblib.dump(X_train_processed, '../data/processed/X_train.pkl')\n",
    "joblib.dump(X_test_processed, '../data/processed/X_test.pkl')\n",
    "joblib.dump(y_train, '../data/processed/y_train.pkl')\n",
    "if y_test is not None:\n",
    "    joblib.dump(y_test, '../data/processed/y_test.pkl')\n",
    "\n",
    "# Save preprocessing pipeline for deployment\n",
    "joblib.dump(preprocessing_pipeline, '../models/preprocessing_pipeline.pkl')\n",
    "\n",
    "print(\"✅ All processed data saved!\")\n",
    "print(f\"  - X_train: {X_train_processed.shape}\")\n",
    "print(f\"  - X_test: {X_test_processed.shape}\")\n",
    "print(f\"  - y_train: {y_train.shape}\")\n",
    "if y_test is not None:\n",
    "    print(f\"  - y_test: {y_test.shape}\")\n",
    "print(f\"  - Pipeline: saved to models/\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Final Data Quality Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Data quality metrics\n",
    "print(\"=\"*80)\n",
    "print(\"📊 FINAL DATA QUALITY REPORT\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "print(f\"\\n📁 Original vs Processed:\")\n",
    "print(f\"  Original: {X_train_raw.shape[1]} features\")\n",
    "print(f\"  Processed: {X_train_processed.shape[1]} features\")\n",
    "print(f\"  Reduction: {X_train_raw.shape[1] - X_train_processed.shape[1]} features removed\")\n",
    "\n",
    "print(f\"\\n🔍 Missing Values (After Processing):\")\n",
    "print(f\"  Total missing: {X_train_processed.isna().sum().sum()}\")\n",
    "print(f\"  Missing percentage: {X_train_processed.isna().sum().sum() / (X_train_processed.shape[0] * X_train_processed.shape[1]) * 100:.4f}%\")\n",
    "\n",
    "print(f\"\\n📊 Feature Diversity:\")\n",
    "print(f\"  Original features: {X_train_raw.shape[1]}\")\n",
    "print(f\"  Engineered features added: {X_train_processed.shape[1] - X_train_raw.shape[1]}\")\n",
    "print(f\"  Final feature set: {X_train_processed.shape[1]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## ✅ Phase 2 Complete!\n",
    "\n",
    "### Summary of Completed Work\n",
    "\n",
    "1. **Missing Value Treatment**:\n",
    "   - Created missing indicators for high-missing features\n",
    "   - Applied MICE imputation for medium missing\n",
    "   - Used median imputation for low missing\n",
    "\n",
    "2. **Histogram Feature Engineering**:\n",
    "   - Extracted mean, variance, skewness, kurtosis\n",
    "   - Calculated peak location and entropy\n",
    "   - Computed IQR for distribution spread\n",
    "\n",
    "3. **Counter Feature Engineering**:\n",
    "   - Created cumulative sums\n",
    "   - Calculated rates of change\n",
    "   - Generated max/min ratios\n",
    "\n",
    "4. **Feature Transformation**:\n",
    "   - Applied Yeo-Johnson transformation to skewed features\n",
    "   - Scaled with RobustScaler (outlier-resistant)\n",
    "\n",
    "5. **Feature Selection**:\n",
    "   - Removed low variance features\n",
    "   - Selected top 100 features using mutual information\n",
    "\n",
    "6. **Pipeline Creation**:\n",
    "   - Built complete preprocessing pipeline\n",
    "   - Saved for production deployment\n",
    "\n",
    "### Next Steps: Phase 3 - Model Development\n",
    "\n",
    "We'll now proceed to:\n",
    "1. Train multiple models (LightGBM, XGBoost, CatBoost, Random Forest, Logistic Regression)\n",
    "2. Optimize hyperparameters with Optuna\n",
    "3. Implement business-cost-aware optimization\n",
    "4. Select optimal threshold based on cost matrix\n",
    "5. Compare models with comprehensive metrics\n",
    "\n",
    "---\n",
    "\n",
    "*End of Feature Engineering Notebook*"
   ]
  }
 ]
}